# HookLab MIE Recognition v0.3 — ruta móvil

Esta libreta permite ejecutar desde el celular la cadena candidata `audio → M + H + T → reconocimiento`. La salida permanece en `D0_EXPLORATORY`, con `scientific_d_unlocked=false`. La sesión de Colab es temporal y la v0.3 aún requiere regresión histórica y multicaso antes de convertirse en baseline.

In [ ]:
# PASO 1 — Instalar HookLab y los motores abiertos (optimizado para Colab)
import importlib.util, os, pathlib, subprocess, sys, time
ROOT = pathlib.Path('/content/hooklab-time')
REF = 'codex/mie-recovery-v0.3'
def run(label, command, timeout=600):
    print(f'▶ {label}', flush=True)
    started = time.time()
    subprocess.run(command, check=True, timeout=timeout)
    print(f'✓ {label} · {time.time()-started:.0f} s', flush=True)
print('Entorno:', sys.version.split()[0], flush=True)
if not ROOT.exists():
    run('Descargar HookLab', ['git','clone','--depth','1','--branch',REF,'https://github.com/basspauloandres-svg/hooklab-time.git',str(ROOT)], 180)
os.chdir(ROOT)
run('Preparar audio del sistema', ['apt-get','update','-qq'], 180)
run('Instalar FFmpeg', ['apt-get','install','-y','-qq','ffmpeg'], 180)
import torch, torchaudio
print('✓ PyTorch de Colab reutilizado ·', torch.__version__, flush=True)
run('Instalar Demucs sin reemplazar PyTorch', [sys.executable,'-m','pip','install','--progress-bar','off','--no-deps','demucs==4.0.1'], 300)
run('Instalar dependencias musicales', [sys.executable,'-m','pip','install','--progress-bar','off','dora-search','einops','julius','lameenc','openunmix','pyyaml','tqdm','basic-pitch==0.4.0','librosa==0.11.0','soundfile==0.13.1','numpy<2','scipy','onnxruntime'], 600)
for module in ('demucs','basic_pitch','librosa','onnxruntime'):
    if importlib.util.find_spec(module) is None: raise RuntimeError('Falta el módulo '+module)
print('INSTALACIÓN LISTA · rama', REF)

In [ ]:
# PASO 2 — Cargar un audio autorizado desde el celular
from google.colab import files
import hashlib, pathlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Carga exactamente un archivo de audio')
original_name, audio_bytes = next(iter(uploaded.items()))
INPUT = pathlib.Path('/content/mie_mobile_input')
INPUT.write_bytes(audio_bytes)
SOURCE_SHA256 = hashlib.sha256(audio_bytes).hexdigest()
print('AUDIO LISTO ·', original_name, '· SHA-256', SOURCE_SHA256)

In [ ]:
# PASO 3 — Ejecutar separación, melodía, armonía, beat, razonamiento y resíntesis
import base64, json, os, pathlib, shutil, subprocess, sys, time
sys.path.insert(0, str(ROOT))
from mie_core.mie_octave_plane_resolver import resolve_event_octaves
from mie_core.mie_recovery_pipeline import apply_reasoning
from mie_core.mie_recognition_contract import normalize
from mie_core.mie_recovery_resynthesis import render
WORK = pathlib.Path('/content/mie_mobile_work')
if WORK.exists(): shutil.rmtree(WORK)
(WORK/'result').mkdir(parents=True)
wav = WORK/'source.wav'
subprocess.run(['ffmpeg','-y','-i',str(INPUT),'-ar','44100','-ac','2',str(wav)],check=True)
subprocess.run([sys.executable,'-m','demucs','-n','htdemucs','-o',str(WORK/'stems'),str(wav)],check=True)
subprocess.run([sys.executable,str(ROOT/'mie_core/run_mie_core.py'),'--audio',str(wav),'--stems',str(WORK/'stems'),'--output',str(WORK/'result')],check=True)
raw = json.loads((WORK/'result/MIE_CORE_MHT_v0_2.json').read_text())
raw['notes'] = resolve_event_octaves(raw.get('notes', []))
analysis_id = 'HL-COLAB-' + str(int(time.time()))
reasoned = apply_reasoning(raw, analysis_id=analysis_id)
output_wav = WORK/'result/MIE_RECOGNITION_MHT_v0_3.wav'
resynthesis = render(reasoned, output_wav)
result = normalize(reasoned, session_id=analysis_id, reference_sha256=SOURCE_SHA256, sensor_version='MIE_CORE_v0.2+RECOVERY_v0.3', ai_provenance=reasoned.get('ai_provenance'))
if result['status'] != 'PASS': raise RuntimeError(result)
result['resynthesis'] = resynthesis
result['source_name'] = original_name
result_path = WORK/'result/MIE_RECOGNITION_v0_3.json'
result_path.write_text(json.dumps(result,indent=2,ensure_ascii=False))
print('RECONOCIMIENTO LISTO · M',len(result['transcription']['melody_events']),'· H',len(result['transcription']['harmony_states']),'· T',len(result['transcription']['beat_events']))

In [ ]:
# PASO 4 — Escuchar la reconstrucción y descargar resultados
from IPython.display import Audio, display
from google.colab import files
import shutil
display(Audio(str(output_wav)))
package = shutil.make_archive('/content/MIE_RECOGNITION_v0_3','zip',WORK/'result')
print('Pulsa aceptar cuando el teléfono solicite descargar el ZIP.')
files.download(package)

## Evaluación del productor

Escucha la reconstrucción y registra por separado: continuidad y octava de la melodía, correspondencia de la armonía, estabilidad del pulso y reconocimiento global de la obra. Una valoración auditiva favorable conserva el artefacto como candidato; la promoción científica requiere calibración independiente.